In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 샘플링과 이산 푸리에 변환 (DFT)

## 학습 목표

1. 연속 신호를 샘플링할 때 샘플링 주파수 $f_s$의 역할을 이해한다
2. 나이퀴스트 정리 $f_s > 2 f_{\max}$ 의 의미와 aliasing을 실험으로 관찰한다
3. DFT 정의식을 직접 구현하여 `np.fft.fft` 와 수치적으로 일치함을 확인한다
4. NumPy FFT 함수군(`fft`/`ifft`, `rfft`/`irfft`, `fftfreq`/`rfftfreq`, `fftshift`)을 구분하여 사용할 수 있다
5. 공통 헬퍼 `utils.fft`로 단측 진폭 스펙트럼을 한 줄로 얻는 법을 익힌다
6. cos 신호의 양측/단측 스펙트럼, 샘플링 파라미터(fs, T)가 스펙트럼에 주는 영향, 고조파 구조를 FFT로 분석한다
7. 주파수 영역 마스킹 기반 필터링을 IFFT로 복원하는 파이프라인을 이해한다
8. 실제 측정 데이터(csv)를 불러와 스펙트럼을 해석한다

## 환경 설정

이 노트북에서는 다음 네 가지만 임포트한다.
- `numpy` : 배열 연산과 FFT
- `matplotlib.pyplot` : 시각화 (스타일 수정 없이 기본값 사용)
- `scipy.signal.find_peaks` : Part C 에서 피크 자동 검출
- `utils.fft` : 실습용 단측 진폭 스펙트럼 헬퍼

색상은 matplotlib 기본 사이클의 `'C0'`, `'C1'`, `'C2'`, `'C3'` 을 직접 지정한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from utils import fft

---

# Part A. 샘플링

컴퓨터가 연속 신호 $x(t)$를 다루려면 일정 간격 $T = 1/f_s$마다 값을 읽어 **이산 수열** $x[n] = x(nT)$로 변환해야 한다. 이 과정을 **샘플링**이라고 한다.

- 샘플링 주파수: $f_s$ (Hz, 초당 샘플 수)
- 샘플 간격: $T = 1/f_s$ (초)
- $N$개의 샘플 → 신호 길이 `duration = N/fs`

## 실습 1. 연속 신호의 샘플링 기본

$f_0 = 3$ Hz 의 cos 신호를 $f_s = 30$ Hz 로 샘플링한다. 연속 신호는 부드러운 실선으로, 이산 샘플은 `stem`으로 표시한다.

In [ ]:
f0 = 3            # 신호 주파수 (Hz)
fs = 30           # 샘플링 주파수 (Hz)
duration = 1.0    # 신호 길이 (초)

# 연속 신호 근사 (매우 조밀한 샘플링)
t_cont = np.linspace(0, duration, 5000)
x_cont = np.cos(2*np.pi*f0*t_cont)

# 샘플링
N = int(fs * duration)
n = np.arange(N)
t = n / fs
x = np.cos(2*np.pi*f0*t)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_cont, x_cont, color='C0', alpha=0.5, label=f'Continuous Signal {f0} Hz')
ax.stem(t, x, linefmt='C1-', markerfmt='C1o', basefmt=' ', label=f'Samples ($f_s$={fs} Hz, N={N})')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Sampling of Continuous Signal')
ax.legend()
plt.tight_layout()
plt.show()

- $f_s = 30$ Hz 는 나이퀴스트 조건 $f_s > 2 f_0 = 6$ Hz 을 충분히 만족한다.
- 샘플 간격 $T = 1/30 \approx 0.033$ s 이고 1초 동안 30개의 샘플이 생긴다.
- 샘플만 보더라도 원래 3 Hz cos 신호의 형태가 그대로 보인다.

### 직접 해보기
1. `fs = 10` 으로 낮추면 샘플 수는 얼마가 되는가? 여전히 원신호 형태가 보이는가?
2. `f0 = 3` 을 `f0 = 20` 으로 바꾸면 어떻게 되는가? ($f_s = 30$ 이면 나이퀴스트 위반)

## 실습 2. Nyquist 주파수와 aliasing

**나이퀴스트 정리**: $f_s > 2 f_{\max}$ 이어야 원 신호를 복원할 수 있다. 이 조건이 위반되면 고주파 성분이 낮은 주파수로 **접혀(alias)** 보인다.

$$ f_{\text{alias}} = | f_0 - k\,f_s |, \quad k = \text{round}(f_0 / f_s) $$

$f_0 = 80$ Hz 신호를 두 가지 샘플링 주파수로 비교한다.

| $f_s$ (Hz) | $f_s/2$ | 조건 | 결과 |
|:---:|:---:|:---:|:---|
| 200 | 100 | $100 > 80$ 충족 | 정상 샘플링 |
| 100 | 50  | $50 < 80$ 위반 | 에일리어스 $|80-100| = 20$ Hz |

In [ ]:
f0 = 80
duration = 0.1
t_cont = np.linspace(0, duration, 8000)
x_cont = np.cos(2*np.pi*f0*t_cont)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for ax, fs in zip(axes, [200, 100]):
    N = int(fs * duration)
    t = np.arange(N) / fs
    x = np.cos(2*np.pi*f0*t)

    # alias 주파수 계산
    f_alias = f0 % fs
    if f_alias > fs/2:
        f_alias = fs - f_alias

    ax.plot(t_cont*1000, x_cont, color='C0', alpha=0.4, label=f'Original {f0} Hz')
    ax.stem(t*1000, x, linefmt='C1-', markerfmt='C1o', basefmt=' ', label='Samples')

    if fs/2 < f0:
        status = 'Nyquist violated'
        x_alias = np.cos(2*np.pi*f_alias*t_cont)
        ax.plot(t_cont*1000, x_alias, '--', color='C2',
                label=f'Alias {f_alias:.0f} Hz')
    else:
        status = 'Nyquist OK'

    ax.set_title(f'$f_s$ = {fs} Hz — {status}')
    ax.set_ylabel('Amplitude')
    ax.set_ylim([-1.4, 1.4])
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

- 위 패널 ($f_s = 200$): 나이퀴스트 여유 충분 → 샘플이 80 Hz cos 를 충실히 추적.
- 아래 패널 ($f_s = 100$): 샘플만 놓고 보면 20 Hz cos 신호와 구분 불가능. 이것이 **aliasing** 이다.
- aliasing 이 발생하면 스펙트럼에서도 실제로 존재하지 않는 20 Hz 피크가 나타난다 (뒤의 Part B 에서 확인).

### 직접 해보기
1. `fs = 160`(충족 경계, $f_s/2 = 80 = f_0$)으로 실행하면 샘플 값이 어떻게 되는가?
2. `f0 = 60, fs = 100` 에서 에일리어스 주파수를 계산하고 그래프로 확인한다.

## 실습 3. 샘플링 정리 체감 — $f_s$ 를 단계적으로 낮추기

$f_{\max} = 120$ Hz 인 합성 신호를 세 가지 $f_s$ 로 샘플링하고 주파수 영역까지 비교한다.

$$x(t) = \cos(2\pi\cdot 50 t) + 0.7\cos(2\pi\cdot 120 t)$$

| $f_s$ | $f_s/2$ | 조건 | 예상 결과 |
|:---:|:---:|:---:|:---|
| 500 | 250 | 충족 (여유 130 Hz) | 50, 120 Hz 피크 선명 |
| 300 | 150 | 충족 (여유 30 Hz)   | 50, 120 Hz 피크 |
| 200 | 100 | 위반                | 120 Hz → 80 Hz 로 접힘 |

> 아래 스펙트럼은 `utils.fft` 헬퍼로 계산한다. 이 함수의 내부 정의와 사용 규약은 Part B-3 에서 자세히 다룬다.

In [ ]:
f_max = 120
duration = 0.5  # 전체 신호 길이 (초)

# 원신호 (연속 파형) — 좌측 40 ms 구간에 겹쳐 그림
t_cont = np.linspace(0, 0.04, 4000)
x_cont = np.cos(2*np.pi*50*t_cont) + 0.7*np.cos(2*np.pi*120*t_cont)

fig, axes = plt.subplots(3, 2, figsize=(14, 8))

for row, fs in enumerate([500, 300, 200]):
    N = int(fs * duration)
    t = np.arange(N) / fs
    x = np.cos(2*np.pi*50*t) + 0.7*np.cos(2*np.pi*120*t)

    # (좌) 원신호 + 샘플링 결과
    axes[row, 0].plot(t_cont*1000, x_cont, color='C0', alpha=0.4, label='Original')
    axes[row, 0].plot(t*1000, x, '-o', color='C1', markersize=4, linewidth=1.0, label='Samples')
    axes[row, 0].set_xlim([0, 40])
    axes[row, 0].set_ylim([-2.0, 2.0])
    axes[row, 0].set_title(f'$f_s$={fs} Hz — Time Domain (first 40 ms, N={int(fs*0.04)} samples)')
    axes[row, 0].set_xlabel('Time (ms)')
    axes[row, 0].set_ylabel('Amplitude')
    axes[row, 0].legend(loc='upper right', fontsize=8)

    # (우) 단측 스펙트럼
    freq, amp = fft(x, fs)
    axes[row, 1].plot(freq, amp, color='C1', lw=1.2)
    ok = fs/2 > f_max
    title = 'No aliasing' if ok else 'Aliasing (120 Hz -> 80 Hz)'
    axes[row, 1].set_title(f'$f_s$={fs} Hz — Spectrum | {title}')
    axes[row, 1].set_xlabel('Frequency (Hz)')
    axes[row, 1].set_ylabel('Amplitude')
    axes[row, 1].set_xlim([0, 260])
    axes[row, 1].set_ylim([0, 1.3])
    axes[row, 1].axvline(fs/2,   color='C0', ls=':',  alpha=0.7, label=f'Nyquist $f_s/2$ = {fs/2:.0f} Hz')
    axes[row, 1].axvline(f_max,  color='C3', ls='--', alpha=0.6, label=f'$f_{{\max}}$ = {f_max} Hz')
    if not ok:
        axes[row, 1].axvline(fs - 120, color='C2', ls='-.', alpha=0.7, label=f'Alias of 120 Hz @ {fs-120:.0f} Hz')
    axes[row, 1].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

- $f_s = 500, 300$ 은 모두 나이퀴스트 조건을 만족하므로 50 Hz 와 120 Hz 에 각각 올바른 진폭의 피크가 나타난다.
- $f_s = 200$ 에서 120 Hz 성분은 $|120 - 200| = 80$ Hz 로 접혀서 나타난다. 스펙트럼만 봐서는 80 Hz 에 실제 성분이 있는지, 120 Hz 가 접힌 것인지 구별할 수 없다.

### 직접 해보기
1. 신호에 `+ 0.5*np.cos(2*np.pi*80*t)` 성분을 추가한다. $f_s = 200$ 에서 80 Hz 피크는 두 성분이 더해져 높아지는가 낮아지는가? (위상에 따라 다름)
2. $f_s = 250$ 에서 실행하면 120 Hz 피크가 살아남지만 여유가 5 Hz 뿐이다. 스펙트럼이 얼마나 깔끔한지 확인한다.

---

# Part B. 이산 푸리에 변환 (DFT)

A-3 에서 본 스펙트럼은 내부적으로 DFT 로 계산된 결과이다. 이제 그 정의와 구현을 정면으로 다룬다.

샘플링으로 얻은 이산 수열 $x[n]$ ($n = 0, 1, \ldots, N-1$)의 주파수 스펙트럼을 유한 합으로 계산하는 것이 **이산 푸리에 변환(DFT)** 이다.

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j 2\pi k n / N}, \qquad k = 0, 1, \ldots, N-1$$

빈 번호 $k$ 는 주파수 $f_k = k\,f_s/N$ 에 대응한다. 주파수 해상도은 $\Delta f = f_s/N = 1/T$ 이다.

> **공지**: 실습 번호는 노트북 전체에서 연속 (Part A = 1~3, Part B = 4~9, Part C = 10~11). 이어지는 Part B 첫 실습은 `실습 4` 부터 시작한다.

## 실습 4. 공통 테스트 신호

Part B 전체에서 사용할 간단한 합성 신호:
$x(t) = \cos(2\pi f_0 t) + 0.5\,\cos(2\pi f_1 t)$

`fs=40`, `N=10`, `f_0=4` Hz, `f_1=12` Hz. $\Delta f = f_s/N = 4$ Hz 라서 두 주파수가 bin 에 정확히 정렬 (누설 없음).

In [ ]:
fs, N = 40, 10
f0, f1 = 4, 12
n = np.arange(N)
t = n / fs
x = np.cos(2*np.pi*f0*t) + 0.5*np.cos(2*np.pi*f1*t)
print(f'fs={fs} Hz, N={N}, T={N/fs:.3f} s, df={fs/N:.2f} Hz')

**관찰**: $x[n] = \cos(2\pi \cdot 4 t) + 0.5\cos(2\pi \cdot 12 t)$, $N=10$, $fs=40$ Hz 의 기본 설정. 이후 실습에서 이 신호의 DFT/FFT/단측-양측 스펙트럼을 반복적으로 사용한다.

## 실습 5. DFT 공식 직접 구현 (O(N²))

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j\,2\pi k n / N}, \qquad k = 0, 1, \ldots, N-1$$

행렬 $W_{k,n} = e^{-j\,2\pi k n / N}$ 를 만들어 $X = W\, x$ 로 한 번에 계산.

In [ ]:
def dft_manual(x, fs):
    """DFT 공식을 행렬 곱으로 직접 구현 (O(N^2)).

    주파수 인덱스는 fftfreq 관례로 반환: k < ceil(N/2) 는 양의 주파수,
    k >= ceil(N/2) 는 Nyquist 이상이므로 음의 주파수(alias) 로 매핑한다.

    Returns
    -------
    freq : ndarray (N,)  -fs/2 ~ +fs/2 (fftfreq 관례, raw 순서)
    X    : ndarray (N,)  복소 스펙트럼
    """
    x = np.asarray(x, dtype=complex)
    N = len(x)
    n = np.arange(N)                    # 시간 인덱스 (열)
    k = np.arange(N).reshape(N, 1)      # 주파수 인덱스 (행)
    W = np.exp(-1j * 2 * np.pi * k * n / N)   # W[k, n]
    X = W @ x
    # k >= ceil(N/2) 는 Nyquist 초과이므로 음 주파수 alias 로 라벨링
    kk = np.arange(N)
    freq = np.where(kk < (N + 1) // 2, kk, kk - N) * fs / N
    return freq, X

freq_manual, X_manual = dft_manual(x, fs)

print('k         :', np.arange(len(x)))
print('freq (Hz) :', freq_manual)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.stem(freq_manual, np.abs(X_manual), linefmt='C0-', markerfmt='C0o', basefmt=' ')
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('|X|')
ax.set_title('dft_manual — magnitude spectrum (fftfreq convention)')
fig.tight_layout(); plt.show()

**관찰**: $W_{k,n} = e^{-j 2\pi kn/N}$ 행렬을 구성하고 `W @ x` 로 DFT 를 직접 계산했다. 반환 주파수 축은 `np.fft.fftfreq` 관례 (인덱스 $k \geq \lceil N/2 \rceil$ 는 음의 주파수로 매핑).

## 실습 6. NumPy FFT 함수군 — 빠른 대안과 도우미들

DFT 공식 구현은 옳지만 O(N²) 로 느리다. NumPy 는 동일 결과를 **O(N log N)** 으로 얻는 **FFT** 를 제공하고, 주파수 축 관련 도우미도 함께 제공한다.

| 함수 | 역할 |
|---|---|
| `np.fft.fft(x)` / `np.fft.ifft(X)` | FFT / 역변환 (복소 입출력) |
| `np.fft.fftfreq(N, d=1/fs)` | 주파수 축 (k 순서) |
| `np.fft.fftshift(X)` | k=0 을 중앙으로 재배열 (양측 스펙트럼 보기 편함) |
| `np.fft.rfft(x)` / `np.fft.irfft(X, n=N)` | 실수 입력 전용 — 단측만 계산 |

먼저 `fft` 와 `dft_manual` 결과가 수치적으로 동일한지 확인.

In [ ]:
X_fft    = np.fft.fft(x)
freq_fft = np.fft.fftfreq(N, d=1/fs)

# sample-wise 비교 (raw order)
print('k  |   freq_manual  freq_fftfreq   |   X_manual                    X_fft')
print('-' * 92)
for kk in range(N):
    fm, ff = freq_manual[kk], freq_fft[kk]
    xm, xf = X_manual[kk], X_fft[kk]
    print(f'{kk:2d} |   {fm:+8.2f}     {ff:+8.2f}   |   '
          f'{xm.real:+8.3f}{xm.imag:+7.3f}j    {xf.real:+8.3f}{xf.imag:+7.3f}j')

# B-1 과 직접 비교: 좌 = dft_manual, 우 = np.fft.fft  (둘 다 Frequency 축)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
ax[0].stem(freq_manual, np.abs(X_manual), linefmt='C0-', markerfmt='C0o', basefmt=' ')
ax[0].set_xlabel('Frequency (Hz)'); ax[0].set_ylabel('|X|')
ax[0].set_title('dft_manual')
ax[1].stem(freq_fft, np.abs(X_fft), linefmt='C1-', markerfmt='C1o', basefmt=' ')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_title('np.fft.fft')
fig.tight_layout(); plt.show()
max_diff = np.max(np.abs(X_manual - X_fft))
print(f'max abs diff : {max_diff:.2e}')
print(f'np.allclose  : {np.allclose(X_manual, X_fft)}')


**관찰**: `max abs diff` 는 기계 정밀도 수준 (∼1e-14), `np.allclose=True` — 두 구현의 결과가 **수치적으로 완전히 동일**. FFT 는 복잡도만 $O(N^2)\to O(N\log N)$ 으로 줄인다.

## 실습 7. `fftshift` 전/후 — 주파수 축 재배열

`np.fft.fft` 반환 순서는 `k=0..N-1`, 즉 **0 ~ fs** 한 바퀴를 돈 것. 실제로는 나이퀴스트 이상 k 는 **음의 주파수**를 의미하므로, `fftshift` 로 **-fs/2 ~ +fs/2** 순으로 재배열하면 직관적이다.

아래 두 출력의 순서 차이를 비교하라.

> Note: 이 `|X|` 값은 아직 `/N` 정규화 전의 원시 스펙트럼이다. B-4 에서 소개할 `fft_full` 은 `/N` 정규화까지 포함한다.

In [ ]:
freq_raw     = np.fft.fftfreq(N, d=1/fs)
freq_shifted = np.fft.fftshift(freq_raw)
X_shifted    = np.fft.fftshift(X_fft)

print('BEFORE fftshift  (k=0..N-1):')
print(f'  freq = {freq_raw}')
print(f'  |X|  = {np.round(np.abs(X_fft), 3)}')
print()
print('AFTER  fftshift  (-fs/2..+fs/2):')
print(f'  freq = {freq_shifted}')
print(f'  |X|  = {np.round(np.abs(X_shifted), 3)}')

# 같은 데이터지만 배열 순서만 다름을 그래프로 확인
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].stem(freq_raw, np.abs(X_fft), linefmt='C0-', markerfmt='C0o', basefmt=' ')
ax[0].set_xlabel('Frequency (Hz)  —  raw order (0..fs)'); ax[0].set_ylabel('|X|')
ax[0].set_title('BEFORE fftshift')
ax[1].stem(freq_shifted, np.abs(X_shifted), linefmt='C2-', markerfmt='C2o', basefmt=' ')
ax[1].set_xlabel('Frequency (Hz)  —  shifted (-fs/2..+fs/2)'); ax[1].set_ylabel('|X|')
ax[1].set_title('AFTER fftshift')
fig.tight_layout(); plt.show()

# 같은 데이터, 배열 순서만 다르다는 사실 코멘트
print()
print('Note: Both outputs are the same DFT; fftshift only reorders the array.')

**관찰**: `fftshift` 로 DC 를 중앙에 배치하면 양측 스펙트럼의 대칭성 (실수 신호 $\Rightarrow |X(-f)| = |X(+f)|$) 이 직관적으로 드러난다.

## 실습 8. DFT 결과의 스케일링 — 물리적 진폭 얻기

`np.fft.fft` 의 출력 `|X|` 자체는 **N (샘플 수)** 에 비례하여 커진다. 물리적 진폭으로 해석하려면 적절히 스케일링해야 한다.

같은 신호 $x(t) = 2\cos(2\pi \cdot 4 \cdot t)$ 를 두 가지 N 으로 샘플링한 결과를 세 단계로 본다.

1. **raw `|X|`**: 피크 높이가 N 에 따라 커진다.
2. **`|X|/N`**: 양측 스펙트럼에서 피크가 원 신호 진폭의 **1/2** 로 나옴 (실수 신호의 켤레 대칭으로 에너지가 $\pm f$ 에 분할).
3. **`2|X|/N` (단측 진폭)**: 음 주파수 에너지를 양 주파수 쪽으로 합쳐 **원 신호 진폭과 일치**. `utils.fft` 가 이 방식을 사용.

In [ ]:
fs = 40
A_signal = 2.0          # 실제 신호 진폭
f0 = 4

N_small, N_large = 10, 100
x_small = A_signal * np.cos(2*np.pi*f0*np.arange(N_small)/fs)
x_large = A_signal * np.cos(2*np.pi*f0*np.arange(N_large)/fs)

# (a) raw |X| (양측, fftshift)
X_small = np.abs(np.fft.fftshift(np.fft.fft(x_small)))
X_large = np.abs(np.fft.fftshift(np.fft.fft(x_large)))
freq_small = np.fft.fftshift(np.fft.fftfreq(N_small, 1/fs))
freq_large = np.fft.fftshift(np.fft.fftfreq(N_large, 1/fs))

# (b) |X|/N (양측 정규화)
X_small_norm = X_small / N_small
X_large_norm = X_large / N_large

# (c) 2|X|/N 단측 (utils.fft 방식)
f_1s_small, A_1s_small = fft(x_small, fs)
f_1s_large, A_1s_large = fft(x_large, fs)

fig, axes = plt.subplots(3, 2, figsize=(13, 8))

# Row 0: raw
axes[0, 0].stem(freq_small, X_small, linefmt='C0-', markerfmt='C0o', basefmt=' ')
axes[0, 0].set_title(f'(a1) N={N_small}  raw |X|   peak = {X_small.max():.1f}')
axes[0, 1].stem(freq_large, X_large, linefmt='C0-', markerfmt='C0o', basefmt=' ')
axes[0, 1].set_title(f'(a2) N={N_large}  raw |X|   peak = {X_large.max():.1f}')
for ax in axes[0]: ax.set_ylabel('|X|')

# Row 1: |X|/N
axes[1, 0].stem(freq_small, X_small_norm, linefmt='C1-', markerfmt='C1o', basefmt=' ')
axes[1, 0].set_title(f'(b1) N={N_small}  |X|/N   peak = {X_small_norm.max():.2f}')
axes[1, 1].stem(freq_large, X_large_norm, linefmt='C1-', markerfmt='C1o', basefmt=' ')
axes[1, 1].set_title(f'(b2) N={N_large}  |X|/N   peak = {X_large_norm.max():.2f}')
for ax in axes[1]: ax.set_ylabel('|X|/N')

# Row 2: 단측 2|X|/N
axes[2, 0].stem(f_1s_small, A_1s_small, linefmt='C2-', markerfmt='C2o', basefmt=' ')
axes[2, 0].set_title(f'(c1) N={N_small}  one-sided 2|X|/N   peak = {A_1s_small.max():.2f}')
axes[2, 1].stem(f_1s_large, A_1s_large, linefmt='C2-', markerfmt='C2o', basefmt=' ')
axes[2, 1].set_title(f'(c2) N={N_large}  one-sided 2|X|/N   peak = {A_1s_large.max():.2f}')
for ax in axes[2]:
    ax.set_ylabel('Amplitude'); ax.set_xlabel('Frequency (Hz)')

plt.tight_layout(); plt.show()

print(f'Original signal amplitude: {A_signal}')
print(f'(a) raw |X| peak         : N={N_small}: {X_small.max():.2f},  N={N_large}: {X_large.max():.2f}   <- scales with N')
print(f'(b) |X|/N peak           : N={N_small}: {X_small_norm.max():.3f},  N={N_large}: {X_large_norm.max():.3f}   <- two-sided, 1/2 of amplitude')
print(f'(c) 2|X|/N one-sided peak: N={N_small}: {A_1s_small.max():.2f},  N={N_large}: {A_1s_large.max():.2f}   <- matches original amplitude')

## 실습 9. 양측 vs 단측 스펙트럼 + 공통 헬퍼

**실수 신호** 는 켤레 대칭 $X[-k] = X[k]^*$ 이라 양측 스펙트럼이 $\pm f$ 에 대칭 피크.

- 양측 (two-sided): $-f_s/2 \sim +f_s/2$ — 대칭 구조를 그대로 본다.
- 단측 (one-sided): $0 \sim f_s/2$ — 음 주파수는 중복이므로 버리고 진폭을 2 배 보정.

공통 헬퍼 (`utils.py`):
- `utils.fft(x, fs) → (f, A)` — 단측 진폭 스펙트럼 (진폭 보정 완료)
- `utils.fft_full(x, fs) → (f, X)` — 양측 복소 스펙트럼 (`fftshift` + `1/N` 정규화)

아래 셀에서 두 함수의 **내부 구현을 직접 확인**한 후, B-5 부터는 이 두 헬퍼만 사용한다.

In [ ]:
import inspect
from utils import fft, fft_full

# 헬퍼 함수 내부 확인
print('=== utils.fft (one-sided amplitude) ===')
print(inspect.getsource(fft))
print('=== utils.fft_full (two-sided complex) ===')
print(inspect.getsource(fft_full))

In [ ]:
# 공통 테스트 신호에 두 헬퍼 적용
f_2s, X_2s = fft_full(x, fs)   # 양측 복소
f_1s, A_1s = fft(x, fs)        # 단측 진폭

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].stem(f_2s, np.abs(X_2s), linefmt='C0-', markerfmt='C0o', basefmt=' ')
ax[0].set_xlabel('Frequency (Hz)'); ax[0].set_ylabel('|X|')
ax[0].set_title('Two-sided Spectrum  (fft_full)')

ax[1].stem(f_1s, A_1s, linefmt='C1-', markerfmt='C1o', basefmt=' ')
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('Amplitude')
ax[1].set_title('One-sided Amplitude Spectrum  (fft)')
fig.tight_layout(); plt.show()

idx_f0 = np.argmin(np.abs(f_1s - f0))
print(f'f0={f0} Hz peak:  two-sided |X|={np.max(np.abs(X_2s)):.3f},  one-sided A={A_1s[idx_f0]:.3f}')

**관찰**: $f_0 = 4$ Hz 에서 one-sided 진폭 $A \approx 1.0$ (원 cos 진폭과 일치), $f_1 = 12$ Hz 에서 $A \approx 0.5$. 양측 $|X|$ 는 각각 $0.5, 0.25$ 로 **절반** — 켤레대칭으로 에너지가 $\pm f$ 에 분할된 결과.

## 실습 10. 샘플링 주파수와 취득시간이 스펙트럼에 주는 영향

테스트 신호: $x(t) = 2\cos(2\pi f_0 t)$, $f_0 = 0.125$ Hz (진폭 2, 느린 cos).

네 가지 $(f_s, T)$ 조합을 비교:

| Case | $f_s$ | $T$ | $N = f_s T$ | $f_s/2$ | $\Delta f = 1/T$ |
|---|---|---|---|---|---|
| Base       | 1 Hz | 8 s  | 8  | 0.5 Hz | 0.125 Hz |
| fs 증가    | 2 Hz | 8 s  | 16 | 1.0 Hz | 0.125 Hz |
| T  증가    | 1 Hz | 16 s | 16 | 0.5 Hz | 0.0625 Hz |
| 둘 다 증가 | 2 Hz | 16 s | 32 | 1.0 Hz | 0.0625 Hz |

**핵심 원리**: $f_s$ 는 **최대 표현 주파수** ($f_s/2$) 를, $T$ 는 **주파수 해상도** ($\Delta f = 1/T$) 를 결정한다 — 서로 독립.

In [ ]:
def sampling_panel(fs_s, T_s, ax_top, ax_bot, title):
    N_s = int(round(fs_s * T_s))
    n_s = np.arange(N_s)
    t_s = n_s / fs_s
    x_s = 2 * np.cos(2 * np.pi * 0.125 * t_s)
    # 시간 도메인: line + marker (stem 대신)
    ax_top.plot(t_s, x_s, '-o', color='C0', markersize=4, linewidth=1.0)
    ax_top.set_xlim([-0.3, 16.3])
    ax_top.set_ylim([-2.3, 2.3])
    ax_top.set_xlabel('Time (s)'); ax_top.set_ylabel('x[n]')
    ax_top.set_title(title)

    # 주파수 도메인: stem (이산 스펙트럼 강조)
    f_2s, X_2s = fft_full(x_s, fs_s)
    ax_bot.stem(f_2s, np.abs(X_2s), linefmt='C0-', markerfmt='C0o', basefmt=' ')
    ax_bot.set_xlim([-0.05, fs_s / 2 + 0.05])
    ax_bot.set_ylim([0, 2.3])
    ax_bot.set_xlabel('Frequency (Hz)'); ax_bot.set_ylabel('|X(f)|')
    ax_bot.set_title('Zoomed')

configs = [
    (1, 8,  '1 Hz , 8 sec  (Base)'),
    (2, 8,  '2 Hz , 8 sec  (Higher fs)'),
    (1, 16, '1 Hz , 16 sec  (Longer T)'),
    (2, 16, '2 Hz , 16 sec  (Both)'),
]

fig, axes = plt.subplots(4, 2, figsize=(13, 12))
for (fs_s, T_s, ttl), row in zip(configs, axes):
    sampling_panel(fs_s, T_s, row[0], row[1], ttl)
fig.tight_layout(); plt.show()

**관찰**
- Row 1 (Base):   $f_s/2 = 0.5$ Hz, $\Delta f = 0.125$ Hz. 0.125 Hz 피크가 bin 1 에 정확히.
- Row 2 (fs↑):    최대 주파수 영역이 **1.0 Hz 까지 확장**. 피크 위치 불변, $\Delta f$ 도 불변.
- Row 3 (T↑):     최대는 0.5 Hz 그대로, **$\Delta f = 0.0625$ Hz 로 절반**. bin 개수 2 배.
- Row 4 (Both↑):  최대 영역 ↑ + $\Delta f$ ↓ 동시 개선.

**정리**: fs ↔ 최대 주파수, T ↔ 주파수 해상도. 두 파라미터는 독립적으로 제어된다.

## 실습 11. FFT 기반 필터링 (주파수 마스크)

주파수 영역에서 원하지 않는 대역을 0 으로 만들고 역변환하여 필터링. 이상적 (sharp cutoff) 필터.

신호: 50 Hz cos + 가우시안 잡음. FFT 후 $\pm 50$ Hz 주변 $\pm 2$ Hz 대역만 남긴다.

In [ ]:
fs_ex, N_ex = 1000, 1000
t_ex = np.arange(N_ex) / fs_ex
np.random.seed(0)
x_clean = np.cos(2 * np.pi * 50 * t_ex)
x_noisy = x_clean + 0.3 * np.random.randn(N_ex)

X = np.fft.fft(x_noisy)
freqs = np.fft.fftfreq(N_ex, d=1 / fs_ex)
mask = (np.abs(freqs - 50) < 1) | (np.abs(freqs + 50) < 1)
X_filt = X * mask
x_recov = np.real(np.fft.ifft(X_filt))

fig, ax = plt.subplots(3, 1, figsize=(11, 7.5))
ax[0].plot(t_ex, x_noisy, 'C0', lw=0.6)
ax[0].set_xlim([0, 0.2]); ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('x')
ax[0].set_title('(a) Noisy Signal  (cos 50 Hz + Gaussian noise)')

f_n, A_n = fft(x_noisy, fs_ex)
ax[1].plot(f_n, A_n, 'C0')
ax[1].axvline(50, color='C3', ls='--', alpha=0.6)
ax[1].set_xlim([0, 200]); ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('|A|')
ax[1].set_title('(b) Noisy Spectrum  — 50 Hz peak stands out')

ax[2].plot(t_ex, x_clean, 'C1', lw=1.2, label='clean')
ax[2].plot(t_ex, x_recov, 'C2', lw=1.0, ls='--', label='recovered')
ax[2].set_xlim([0, 0.2]); ax[2].set_xlabel('Time (s)'); ax[2].set_ylabel('x')
ax[2].legend(); ax[2].set_title('(c) Clean vs Recovered  (after FFT-mask + IFFT)')
fig.tight_layout(); plt.show()

rmse = np.sqrt(np.mean((x_clean - x_recov) ** 2))
print(f'Recovery RMSE vs clean : {rmse:.4f}')

**관찰**: $\pm 50$ Hz 주변 $\pm 1$ Hz 마스크로 잡음 속 50 Hz cos 성분만 남긴다. `N_ex = 1000`, `fs = 1000` 이라 bin 간격이 정확히 1 Hz 여서 50 Hz bin 에 에너지가 깨끗이 실린다. 복원 RMSE 가 낮게 나오며 (c) 패널에서 clean/recovered 가 거의 일치함을 확인.

---

# Part C. 실제 신호와의 연결

측정 데이터 `data/data_sample_fft.csv` 를 불러와 지금까지 배운 파이프라인을 적용한다.

이 파일은 `index, time(s), value` 의 3개 열을 가지며 `dt = 0.001` s (즉 $f_s = 1000$ Hz) 간격으로 저장되어 있다. 다른 환경에서 파일이 `[time, value]` 2개 열로 제공될 수도 있으므로 아래 셀에서 방어적으로 로드한다.

**흐름**: (1) 시간 신호 확인 → (2) 샘플링 정보 계산 → (3) utils.fft 로 스펙트럼 확인 → (4) 주요 성분 해석.

In [ ]:
data = np.loadtxt('./data/data_sample_fft.csv', delimiter=',', skiprows=1)

# CSV 구조는 환경에 따라 [time, value] 또는 [index, time, value] 일 수 있다
if data.shape[1] == 2:
    t, x = data[:, 0], data[:, 1]
elif data.shape[1] == 3:
    t, x = data[:, 1], data[:, 2]
else:
    raise ValueError(f'Unexpected CSV shape: shape={data.shape}')

dt = t[1] - t[0]
fs = 1.0 / dt       # 실제 시간 간격으로부터 재계산
N = len(x)

print(f'CSV cols  : {data.shape[1]}')
print(f'Number of Samples N    : {N}')
print(f'dt (step) : {dt:.4f} s')
print(f'Sampling Frequency fs : {fs:.1f} Hz')
print(f'Observation T  : {N*dt:.3f} s')
print(f'Frequency Resolution Δf = 1/T = {1/(N*dt):.3f} Hz')

In [ ]:
f, A = fft(x, fs)

# scipy.signal.find_peaks 로 주요 피크 자동 검출
peak_idx, _ = find_peaks(A, height=A.max()*0.1)

fig, axes = plt.subplots(2, 1, figsize=(13, 6))
axes[0].plot(t, x, color='C0', lw=0.8)
axes[0].set_title('(a) Measured Time Signal  data_sample_fft.csv')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

axes[1].plot(f, A, color='C1', lw=1.2)
axes[1].plot(f[peak_idx], A[peak_idx], 'v', color='C3', markersize=8,
             label=f'find_peaks detected ({len(peak_idx)})')
axes[1].set_title('(b) One-sided Amplitude Spectrum  utils.fft(x, fs)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlim([0, fs/2])
axes[1].legend()

plt.tight_layout()
plt.show()

print('Detected peaks (Frequency, Amplitude):')
for k in peak_idx:
    print(f'  f = {f[k]:7.2f} Hz   A = {A[k]:.3f}')

- **Part A → B 의 연결**: 시간 축에서 보는 신호는 $f_s = 1000$ Hz 의 샘플링 결과이다. 나이퀴스트 한계는 500 Hz 이므로 그 이상의 성분은 물리적으로 존재하더라도 스펙트럼에 접혀(alias) 나타날 수 있다.
- **주파수 해상도**: $N = 1000$, $T = 1$ s → $\Delta f = 1$ Hz. 1 Hz 간격으로 성분을 분리할 수 있다.
- **피크 해석**: `scipy.signal.find_peaks(A, height=A.max()*0.1)` 로 자동 검출된 피크는 **60 Hz 단일 성분** (A ≈ 2.0) 으로, 이 데이터의 지배적 주기 성분이다. `height` 임계값을 낮추면 잡음 수준의 추가 피크가 검출될 수 있다.

### 직접 해보기
1. 신호의 앞 500 샘플만 사용하여 FFT 를 다시 계산해보자. $\Delta f$ 가 어떻게 달라지는가?
2. `find_peaks` 의 `height` 임계값을 `A.max()*0.05` 로 낮추면 어떤 피크가 추가로 검출되는가?
3. 검출된 피크 주파수들의 비율을 계산하여 기본 주파수의 고조파 관계인지 확인한다.

---
# 정리

## 핵심 개념

| 개념 | 수식 / 규칙 | 요점 |
|------|------------|------|
| 샘플링 간격 | $T = 1/f_s$ | 초당 $f_s$ 개의 샘플 |
| 나이퀴스트 정리 | $f_s > 2 f_{\max}$ | 위반 시 aliasing 발생 |
| 에일리어스 주파수 | $f_{\text{alias}} = \lvert f_0 - k f_s \rvert$ | $f_s/2$ 이하로 접힘 |
| DFT 정의 | $X[k] = \sum_n x[n] e^{-j 2\pi k n/N}$ | $O(N^2)$ 행렬곱 |
| FFT | `np.fft.fft` | $O(N \log N)$, 결과 동일 |
| 주파수 해상도 | $\Delta f = f_s/N = 1/T$ | $N$ 또는 $T$ 증가 시 향상 |
| 단측 진폭 | $A_k = 2\lvert X[k]\rvert/N$ | DC, Nyquist 는 1배 |
| FFT 함수 규약 | `f, A = fft(x, fs)` | 이 노트북의 공통 헬퍼 |

## 두 개념의 연결

샘플링은 시간 영역의 문제(어떤 간격으로 값을 읽을까), DFT 는 주파수 영역의 문제(이산 샘플을 주파수 성분으로 분해)이다. 두 개념은 다음 식으로 묶인다:

$$\boxed{\;\Delta f = \frac{1}{T} = \frac{f_s}{N}\;}$$

- 관측 시간 $T$ 를 늘린다 → 주파수 해상도 향상
- 샘플링 주파수 $f_s$ 를 올린다 → 최대 분석 가능 주파수 $f_s/2$ 증가
